## Importing the Data

In [20]:
import pandas as pd

collegeScorecard = pd.read_csv(
    "../data/processed/College_Scorecard_cleaned.csv",
    dtype={"CIP_family": str}
)
pseo = pd.read_csv(
    "../data/processed/PSEO_cleaned.csv",
    dtype={"CIP_family": str, "CIP_series": str}
)

pseo_cip = set(pseo["CIP_family"].dropna())
pseo_series = set(pseo["CIP_series"].dropna())

print("CollegeScorecard shape:", collegeScorecard.shape)
print("PSEO shape:", pseo.shape)

CollegeScorecard shape: (26678, 101)
PSEO shape: (18826, 69)


## Basic Analysis before Merge

Finding the unique CIP_family values from each dataset:

In [21]:
cs_cip = set(collegeScorecard["CIP_family"].dropna())
pseo_cip = set(pseo["CIP_family"].dropna())

print("CollegeScorecard unique CIP_family values:", len(cs_cip))
print("PSEO unique CIP_family values:", len(pseo_cip))
print("Overlapping CIP_family values:", len(cs_cip & pseo_cip))

CollegeScorecard unique CIP_family values: 360
PSEO unique CIP_family values: 319
Overlapping CIP_family values: 307


Checking which values are in one dataset but not the other:

In [22]:
only_in_cs = sorted(cs_cip - pseo_cip)
only_in_pseo = sorted(pseo_cip - cs_cip)

print("\nIn CollegeScorecard but not PSEO:", len(only_in_cs))
print(only_in_cs)

print("\nIn PSEO but not CollegeScorecard:", len(only_in_pseo))
print(only_in_pseo)


In CollegeScorecard but not PSEO: 53
['01.82', '03.99', '04.02', '04.08', '04.10', '04.99', '05.99', '12.06', '13.03', '13.07', '14.06', '14.11', '14.22', '14.23', '14.33', '14.37', '14.41', '14.48', '15.14', '16.14', '16.17', '19.04', '19.99', '25.99', '26.04', '26.07', '26.08', '27.99', '28.06', '29.99', '30.08', '30.22', '30.25', '30.29', '30.36', '30.41', '30.44', '30.46', '30.51', '30.52', '30.53', '30.70', '39.99', '40.02', '45.13', '48.00', '50.11', '51.01', '51.04', '51.12', '51.16', '53.02', '61.26']

In PSEO but not CollegeScorecard: 12
['14.03', '14.32', '14.39', '14.40', '14.45', '26.05', '29.03', '33.01', '35.01', '37.01', '45.05', '60.01']


In [23]:
collegeScorecard["has_pseo_match"] = collegeScorecard["CIP_family"].isin(pseo_cip)

match_pct = collegeScorecard["has_pseo_match"].mean() * 100
print(f"\n% of CollegeScorecard rows with a CIP_family found in PSEO: {match_pct:.1f}%")


% of CollegeScorecard rows with a CIP_family found in PSEO: 99.5%


In [24]:
missing_rows = collegeScorecard[~collegeScorecard["has_pseo_match"]]
print("\nTop CIP_family values among unmatched rows:")
print(missing_rows["CIP_family"].value_counts().head(10))


Top CIP_family values among unmatched rows:
CIP_family
30.70    18
16.17    10
13.03     7
51.16     6
48.00     5
39.99     5
27.99     4
30.22     4
26.07     3
16.14     3
Name: count, dtype: int64


## Things to Note:

- Format matches exactly — both use the same `XX.YY` style (e.g. `16.09`), so no reformatting was needed.
- Of our 360 unique CIP_family codes, 307 have a match in PSEO.
- 53 CIP_family codes in our data don't appear in PSEO at all — any program under one of these codes will end up with no earnings data after merging, not because of missing/suppressed values, but because PSEO simply doesn't cover that field of study.
- At the row level, **99.5% of our actual rows** have a CIP_family that exists in PSEO — this is the number that matters more than the code-level count above, since it reflects how much of our real data would survive the merge.

-----
## Possible fix:

- Comparing our CIP codes against PSEO's, 99.51% of our rows get an exact match on the full CIP code. Only 0.47% don't match exactly but could still get a close estimate by matching on the shortened (broader category) CIP code instead. 
- Merge using the full CIP code first, and fall back to the shortened CIP code only for the small number of rows that don't have an exact match. This gets us earnings data for almost every row without losing precision on the ones that already match.


In [25]:
# Row-level match breakdown: family-level match, series-level fallback, or no match at all
collegeScorecard["CIP_series"] = collegeScorecard["CIP_family"].str[:2]

def match_level(cip_family):
    if pd.isna(cip_family):
        return "none"
    if cip_family in pseo_cip:
        return "family"
    elif cip_family[:2] in pseo_series:
        return "series"
    else:
        return "none"

collegeScorecard["cip_match_level"] = collegeScorecard["CIP_family"].apply(match_level)

match_counts = collegeScorecard["cip_match_level"].value_counts()
match_pcts = collegeScorecard["cip_match_level"].value_counts(normalize=True) * 100

print(pd.DataFrame({"rows": match_counts, "pct": match_pcts.round(2)}))

                  rows    pct
cip_match_level              
family           26546  99.51
series             126   0.47
none                 6   0.02


**NOTE:** Looking at this analysis, just 0.02% (6 rows) don't match at all, no matter which version of the code we use.

------
## Further Analysis: Credential and Institutional Level



In [26]:
print(collegeScorecard["CREDLEV"].value_counts())
print(pseo["degree_level"].value_counts())
print(pseo["degree_level_label"].value_counts())

# Real institution + CIP match rate: check if (OPEID6, CIP_family) pairs exist in PSEO
pseo_keys = set(zip(pseo["OPEID6"], pseo["CIP_family"]))
collegeScorecard["has_institution_cip_match"] = collegeScorecard.apply(
    lambda row: (row["OPEID6"], row["CIP_family"]) in pseo_keys, axis=1
)

inst_match_pct = collegeScorecard["has_institution_cip_match"].mean() * 100
print(f"Real institution+CIP match rate: {inst_match_pct:.2f}%")

# Duplicate-key check: are there any (OPEID6, CIP_family) pairs with more than one PSEO row?
dupe_keys = pseo.groupby(["OPEID6", "CIP_family"]).size()
print("(OPEID6, CIP_family) pairs with more than 1 PSEO row:", (dupe_keys > 1).sum())

CREDLEV
1    26678
Name: count, dtype: int64
degree_level
1    10025
2     8322
4      479
Name: count, dtype: int64
degree_level_label
Certificate < 1 year     10025
Certificate 1-2 years     8322
Certificate 2-4 years      479
Name: count, dtype: int64
Real institution+CIP match rate: 17.37%
(OPEID6, CIP_family) pairs with more than 1 PSEO row: 4169


----------------
## PSEO + CS Merge: Summary

**Checked:** CIP code format, CIP code coverage, credential level alignment, and institution-level match rate.

**Findings:**
- CIP codes match well: 99.5% of our rows have a usable match.
- Credential levels already line up (both datasets are certificate-only).
- But matching on the **actual school + program** (not just program) drops coverage to **17.4%**: PSEO simply doesn't report data for most of our schools, likely due to its own privacy-suppression rules.

**Decision for the team:**
| Option | Coverage | Precision |
|---|---|---|
| School + program match | 17.4% | Most accurate |
| Program only | 99.5% | Less precise (national average, not school-specific) |
| Hybrid (school match, fall back to program-only) | 99.5% | Best of both |

**Suggestion:** Hybrid - use the precise number when available, fall back when not, and flag which rows got which.